# 02 Data Preparation for Graph Construction

Preparing data for graph construction by extracting and matching relevant information from RxNorm and OpenFDA datasets, and structuring it for use in the knowledge graph. Explore the medication knowledge graph built from OpenFDA + RxNorm data.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

project_root = os.chdir("..")
sys.path.insert(0, str(project_root))

os.getcwd()

In [ ]:
import random
import pandas as pd
import json
import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame

In [ ]:
from src.knowledge_graph.openfda_data_collection import *
from src.knowledge_graph.rxnorm_data_collection import *
from src.knowledge_graph.build_graph_data import *

## 1 Get drug list sample

In [ ]:
drug_list_all = load_drug_list()

In [ ]:
# FOR DEVELOPMENT: SAMPLE ONLY
sample_size = 10
drug_list = random.sample(list(drug_list_all), sample_size)

In [ ]:
drug_list

## 2 Run OpenFDA data collection

In [ ]:
raw_data = fetch_openfda_raw_data(drug_list)

In [ ]:
raw_path = save_raw_openfda_data(raw_data, sample=True)

In [ ]:
rxcui_data = transform_to_rxcui_keyed(raw_data)

In [ ]:
rxcui_path = save_rxcui_keyed_data(rxcui_data, sample=True)

In [ ]:
rxcui_list_path, rxcui_list = save_rxcui_list(rxcui_data, sample=True)

In [ ]:
#rxcui_list_path = "data/02_intermediate/top_300_rxcuis_sample.txt"

## 3 Run RxNorm data collection

In [ ]:
rxcui_list = load_rxcui_list(rxcui_list_path)

In [ ]:
# Export raw RxNorm data to Parquet files from MySQL database
# Note: This is a one-time operation. The resulting Parquet files are saved in the data/ directory and can be loaded directly in the next step.
# rxnconso_df, rxnrel_df, rxnsat_df = export_raw_parquets()

# Load raw RxNorm data from Parquet files
rxnconso_df, rxnrel_df, rxnsat_df = load_raw_parquets()

In [ ]:
rxnconso_filtered, rxnrel_filtered, rxnsat_filtered = filter_tables(
        rxnconso_df, rxnrel_df, rxnsat_df, rxcui_list
    )

## 3 Build graph data

In [ ]:
# load json
with open("data/02_intermediate/openfda_by_rxcui_sample.json", "r") as f:
    rxcui_data = json.load(f)

In [ ]:
print("Total number of unique RXCUIs in rxcui_data:", len(rxcui_data))

# Keep only first 10 keys of rxcui_data for testing
rxcui_data_sample = {k: rxcui_data[k] for k in list(rxcui_data.keys())[:1]}
rxcui_data_sample

In [ ]:
# openfda_data = load_openfda_data()
openfda_data = rxcui_data_sample.copy()

In [ ]:
openfda_entries = openfda_data['309594']
routes = []
product_types = []
brand_names = []

for openfda_entry in openfda_entries:
    data_dict = openfda_entry.get("data", {})
    routes.extend(data_dict.get("openfda", {}).get("route", []))
    product_types.extend(data_dict.get("openfda", {}).get("product_type", []))
    brand_names.extend(data_dict.get("openfda", {}).get("brand_name", []))

# Standardize and deduplicate the extracted attributes
routes = list(set([route.strip().lower() for route in routes]))
product_types = list(set([product_type.strip().lower() for product_type in product_types]))
brand_names = list(set([brand_name.strip().lower() for brand_name in brand_names]))

print("Routes:", routes)
print("Product Types:", product_types)
print("Brand Names:", brand_names)


In [ ]:
drug_nodes = build_drug_nodes(rxnconso_filtered, openfda_data)

In [ ]:
drug_nodes

In [ ]:
ingredient_nodes = build_ingredient_nodes(rxnrel_df, rxnconso_df)
# ingredient_nodes = build_ingredient_nodes(rxnrel_filtered, rxnconso_filtered)

In [ ]:
ingredient_nodes

In [ ]:
edges = build_edges_from_openfda(openfda_data, drug_nodes, ingredient_nodes)

In [ ]:
edges

In [ ]:
first_entry = openfda_data['309594'][0]
first_entry

In [ ]:
openfda_connector = OpenFDAConnector()


In [ ]:
first_entry_structured = openfda_connector.extract_structured_fields(first_entry['data'])
first_entry_structured

In [ ]:
for dn in drug_nodes.values():
    if dn.get("rxcui") == '309594':
        drug_node = dn

In [ ]:
drug_node

In [ ]:
existing_nodes = {**drug_nodes, **ingredient_nodes}
missing_nodes = create_missing_nodes(edges, existing_nodes)
all_nodes = {**existing_nodes, **missing_nodes}

In [ ]:
save_entities_and_relations(all_nodes, edges)

In [ ]:
G = build_networkx_graph(all_nodes, edges)
save_graph(G)

In [ ]:
# Load graph from JSON
graph_file = "data/03_graph/graph.json"

if not Path(graph_file).exists():
    print(f"Graph file not found: {graph_file}")
    print("Please run scripts 03-05 first to build the graph data.")
else:
    with open(graph_file, "r") as f:
        graph_data = json.load(f)
    
    print(f"✅ Loaded graph data")
    print(f"Nodes: {len(graph_data['nodes'])}")
    print(f"Edges: {len(graph_data['edges'])}")

In [ ]:
# Reconstruct NetworkX graph from JSON
G = nx.DiGraph()

# Add nodes
for node in graph_data['nodes']:
    node_copy = node.copy()
    node_id = node_copy.pop('id')
    G.add_node(node_id, **node_copy)

# Add edges
for edge in graph_data['edges']:
    u = edge['source']
    v = edge['target']
    edge_attrs = {k: v for k, v in edge.items() if k not in ['source', 'target']}
    G.add_edge(u, v, **edge_attrs)

print(f"✅ Built NetworkX graph")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.4f}")


In [ ]:
# Quick static plot: top-N nodes by degree
# Adjust `top_n` to control how many high-degree nodes to display
from pathlib import Path
import json
import matplotlib.pyplot as plt

top_n = 100

# Ensure G exists (reconstructed above); otherwise load saved graph
if "G" not in globals():
    data = json.loads(Path("data/03_graph/graph.json").read_text())
    G = nx.DiGraph()
    for n in data["nodes"]:
        nid = n["id"]
        attrs = {k: v for k, v in n.items() if k != "id"}
        G.add_node(nid, **attrs)
    for e in data["edges"]:
        src, tgt = e["source"], e["target"]
        attrs = {k: v for k, v in e.items() if k not in ("source", "target")}
        G.add_edge(src, tgt, **attrs)

# Select top-N nodes by degree
nodes_by_degree = sorted(G.degree(), key=lambda x: x[1], reverse=True)
keep = [n for n, _ in nodes_by_degree[:top_n]]
G_plot = G.subgraph(keep).copy()

print(f"Plotting top {len(G_plot)} nodes (top_n={top_n}) — original graph nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

# Visual params
type_colors = {"Drug":"tab:blue","Ingredient":"tab:green","Warning":"tab:red",
               "Condition":"tab:purple","SideEffect":"tab:orange"}
colors = [type_colors.get(d.get("type",""), "lightgrey") for _, d in G_plot.nodes(data=True)]
sizes = [100 + 30 * G_plot.degree(n) for n in G_plot.nodes()]

# Layout & draw
pos = nx.spring_layout(G_plot, seed=42, k=0.5, iterations=200)
plt.figure(figsize=(14, 9))
nx.draw_networkx_nodes(G_plot, pos, node_color=colors, node_size=sizes, alpha=0.9)
nx.draw_networkx_edges(G_plot, pos, alpha=0.6, arrowsize=12)

# Show labels for a small number of nodes
if G_plot.number_of_nodes() <= 40:
    labels = {n: d.get("name", n) for n, d in G_plot.nodes(data=True)}
    nx.draw_networkx_labels(G_plot, pos, labels, font_size=8)

plt.title(f"Top-{len(G_plot)} nodes by degree — nodes={G_plot.number_of_nodes()}, edges={G_plot.number_of_edges()}")
plt.axis("off")
plt.show()


In [ ]:
# Interactive Plotly network (top-N nodes) — saves an HTML you can open in Chrome
# Adjust `top_n` to control size. Uses Plotly for hover, pan, zoom, and legend.
import json
from pathlib import Path
import plotly.graph_objects as go
from plotly.offline import plot

top_n = 150
out_html = Path(f"data/03_graph/graph_interactive_top{top_n}.html")

# Ensure G exists
if "G" not in globals():
    data = json.loads(Path("data/03_graph/graph.json").read_text())
    G = nx.DiGraph()
    for n in data["nodes"]:
        nid = n["id"]
        attrs = {k: v for k, v in n.items() if k != "id"}
        G.add_node(nid, **attrs)
    for e in data["edges"]:
        src, tgt = e["source"], e["target"]
        attrs = {k: v for k, v in e.items() if k not in ("source", "target")}
        G.add_edge(src, tgt, **attrs)

# Select top-N nodes by degree (keep their neighbors as context)
nodes_by_degree = sorted(G.degree(), key=lambda x: x[1], reverse=True)
core = [n for n, _ in nodes_by_degree[:top_n]]
# include 1-hop neighbors
keep = set(core)
for n in core:
    keep.update(G.successors(n))
    keep.update(G.predecessors(n))
G_plot = G.subgraph(keep).copy()

print(f"Interactive plot: nodes={G_plot.number_of_nodes()}, edges={G_plot.number_of_edges()} -> saving to {out_html}")

# Compute layout
pos = nx.spring_layout(G_plot, seed=42, k=0.5, iterations=100)

# Edge traces
edge_x = []
edge_y = []
for u, v in G_plot.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scattergl(x=edge_x, y=edge_y,
                          mode='lines',
                          line=dict(width=0.5, color='#888'),
                          hoverinfo='none')

# Node traces per type for legend
type_colors = {"Drug":"#1f78b4","Ingredient":"#33a02c","Warning":"#e31a1c","Condition":"#6a3d9a","SideEffect":"#ff7f00"}
nodes_by_type = {}
for n, d in G_plot.nodes(data=True):
    t = d.get('type', 'Unknown')
    nodes_by_type.setdefault(t, []).append((n, d))

node_traces = []
for t, nodes in nodes_by_type.items():
    x = [pos[n][0] for n, _ in nodes]
    y = [pos[n][1] for n, _ in nodes]
    hover = []
    sizes = []
    for n, d in nodes:
        name = d.get('name', n)
        rxcui = d.get('rxcui', '')
        aliases = ', '.join(d.get('aliases', [])[:3]) if isinstance(d.get('aliases', []), list) else ''
        hover.append(f"{name}<br>type: {t}<br>rxcui: {rxcui}<br>aliases: {aliases}")
        sizes.append(8 + 2 * G_plot.degree(n))

    trace = go.Scattergl(x=x, y=y, mode='markers', name=t,
                         marker=dict(size=sizes, color=type_colors.get(t, '#bdbdbd'), line=dict(width=0.5, color='#333')),
                         text=[d.get('name', n) for n, d in nodes], hovertext=hover, hoverinfo='text')
    node_traces.append(trace)

# Build figure
fig = go.Figure(data=[edge_trace] + node_traces)
fig.update_layout(showlegend=True, title_text=f"Interactive graph (top {top_n} by degree)",
                  hovermode='closest', margin=dict(b=20,l=5,r=5,t=40))
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)

# Save HTML
out_html.parent.mkdir(parents=True, exist_ok=True)
plot(fig, filename=str(out_html), auto_open=False, include_plotlyjs='cdn')
print(f"Saved interactive HTML: {out_html}")

# Optionally display inline (works in notebook if not too large)
try:
    from IPython.display import IFrame
    display(IFrame(str(out_html), width='100%', height=700))
except Exception:
    pass


In [ ]:
import scipy

# Visualize the knowledge graph
pos = nx.spring_layout(G, seed=42, k=0.9)
labels = nx.get_edge_attributes(G, 'label')
plt.figure(figsize=(12, 10))
nx.draw(G, pos, with_labels=True, font_size=10, node_size=700, node_color='lightblue', edge_color='gray', alpha=0.6)
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=8, label_pos=0.3, verticalalignment='baseline')
plt.title('Knowledge Graph')
plt.show()

In [ ]:

# # load saved graph
# graph_path = Path("data/03_graph/graph.json")
# data = json.loads(graph_path.read_text())

# # rebuild NetworkX graph
# G = nx.DiGraph()
# for n in data["nodes"]:
#     nid = n["id"]
#     attrs = {k:v for k,v in n.items() if k!="id"}
#     G.add_node(nid, **attrs)
# for e in data["edges"]:
#     src, tgt = e["source"], e["target"]
#     attrs = {k:v for k,v in e.items() if k not in ("source","target")}
#     G.add_edge(src, tgt, **attrs)

# pyvis network
net = Network(height="750px", width="100%", notebook=True, directed=True)
color_map = {"Drug":"#1f78b4","Ingredient":"#33a02c","Warning":"#e31a1c","Condition":"#6a3d9a","SideEffect":"#ff7f00"}

for node, d in G.nodes(data=True):
    ntype = d.get("type","Unknown")
    net.add_node(node, label=d.get("name", node), title=f"{d.get('name','')}\n{ntype}", color=color_map.get(ntype,"#bdbdbd"))

for u,v,ad in G.edges(data=True):
    title = ad.get("relation", "")
    if ad.get("evidence_text"):
        title += " — " + ad["evidence_text"][:200]
    net.add_edge(u, v, title=title, color="#666")

net.toggle_physics(True)
out_html = "graph.html"
net.show(out_html)
IFrame(out_html, width="100%", height=750, cdn_resources="in_line")

In [ ]:
IFrame(out_html, width="100%", height=750, cdn_resources="in_line")

In [ ]:
# Static NetworkX + Matplotlib plot (paste into notebook)
import json
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt

# Load or reuse existing G
if "G" not in globals():
    data = json.loads(Path("data/03_graph/graph.json").read_text())
    G = nx.DiGraph()
    for n in data["nodes"]:
        nid = n["id"]
        attrs = {k: v for k, v in n.items() if k != "id"}
        G.add_node(nid, **attrs)
    for e in data["edges"]:
        src, tgt = e["source"], e["target"]
        attrs = {k: v for k, v in e.items() if k not in ("source", "target")}
        G.add_edge(src, tgt, **attrs)

# Reduce size for plotting if large
max_nodes = 150
if G.number_of_nodes() > max_nodes:
    nodes_by_degree = sorted(G.degree(), key=lambda x: x[1], reverse=True)
    keep = [n for n, _ in nodes_by_degree[:max_nodes]]
    G_plot = G.subgraph(keep).copy()
else:
    G_plot = G

# Visual parameters
type_colors = {"Drug": "tab:blue", "Ingredient": "tab:green", "Warning": "tab:red",
               "Condition": "tab:purple", "SideEffect": "tab:orange"}
colors = [type_colors.get(d.get("type", ""), "lightgrey") for _, d in G_plot.nodes(data=True)]
sizes = [50 + 20 * G_plot.degree(n) for n in G_plot.nodes()]

# Layout and draw
pos = nx.spring_layout(G_plot, seed=42, k=0.5, iterations=100)
plt.figure(figsize=(12, 8))
nx.draw_networkx_nodes(G_plot, pos, node_color=colors, node_size=sizes, alpha=0.9)
nx.draw_networkx_edges(G_plot, pos, alpha=0.5, arrowsize=10)
if G_plot.number_of_nodes() <= 60:
    labels = {n: d.get("name", n) for n, d in G_plot.nodes(data=True)}
    nx.draw_networkx_labels(G_plot, pos, labels, font_size=8)
plt.title(f"Graph (nodes={G_plot.number_of_nodes()}, edges={G_plot.number_of_edges()})")
plt.axis("off")
plt.show()